# Get GitLab root password

In [ ]:
import subprocess
from pathlib import Path

GITLAB_PASSWORD_PATH = Path("assets/gitlab_password.txt")

def read_password_from_file() -> str | None:
    if not GITLAB_PASSWORD_PATH.exists():
        return None

    password = GITLAB_PASSWORD_PATH.read_text(encoding="utf-8").strip()
    return password or None


def read_password_from_gitlab() -> str:
    result = subprocess.run(
        [
            "docker",
            "compose",
            "exec",
            "-T",
            "gitlab",
            "awk",
            "/^Password:/ {print $2}",
            "/etc/gitlab/initial_root_password",
        ],
        check=True,
        capture_output=True,
        text=True,
    )

    password = result.stdout.strip()
    if not password:
        raise RuntimeError("GitLab root password was not found")

    return password


def save_password(password: str) -> None:
    GITLAB_PASSWORD_PATH.parent.mkdir(parents=True, exist_ok=True)
    GITLAB_PASSWORD_PATH.write_text(f"{password}\n", encoding="utf-8")


def get_gitlab_root_password() -> str:
    password = read_password_from_file()

    if password is not None:
        return password

    password = read_password_from_gitlab()
    save_password(password)
    return password


gitlab_password = get_gitlab_root_password()
print(f'gitlab root password: {gitlab_password}')

# Create GitLab root token

In [ ]:
import subprocess
from pathlib import Path

GITLAB_TOKEN_PATH = Path("assets/gitlab_token.txt")

RAILS_SCRIPT = """
user = User.find_by_username("root")
token = user.personal_access_tokens.create!(
    name: "bootstrap",
    scopes: ["api"],
    expires_at: 365.days.from_now
)
puts token.token
"""


def read_token_from_file() -> str | None:
    if not GITLAB_TOKEN_PATH.exists():
        return None

    token = GITLAB_TOKEN_PATH.read_text(encoding="utf-8").strip()
    return token or None


def read_token_from_gitlab() -> str:
    result = subprocess.run(
        [
            "docker",
            "exec",
            "gitlab",
            "gitlab-rails",
            "runner",
            RAILS_SCRIPT,
        ],
        check=True,
        capture_output=True,
        text=True,
    )

    token = result.stdout.strip()

    if not token:
        raise RuntimeError("GitLab did not return a token")

    return token


def save_gitlab_token(token: str) -> None:
    GITLAB_TOKEN_PATH.parent.mkdir(parents=True, exist_ok=True)
    GITLAB_TOKEN_PATH.write_text(f"{token}\n", encoding="utf-8")


def get_gitlab_token() -> str:
    token = read_token_from_file()

    if token is not None:
        return token

    token = read_token_from_gitlab()
    save_gitlab_token(token)
    return token

gitlab_token = get_gitlab_token()
print(f'gitlab token: {gitlab_token}')

# Log in to the GitLab Container Registry using Docker

In [ ]:
def docker_login(
    registry: str,
    username: str,
    password: str,
) -> None:
    subprocess.run(
        [
            "docker",
            "login",
            registry,
            "--username",
            username,
            "--password-stdin",
        ],
        input=password,
        text=True,
        check=True,
    )

docker_login(
    registry="registry.gitlab.localhost",
    username="root",
    password=gitlab_password,
)

# Create GitLab HTTPS client

In [ ]:
import requests

GITLAB_INTERNAL_URL = "http://gitlab"
GITLAB_EXTERNAL_URL = "https://gitlab.localhost"

PROJECT_ROOT = Path.cwd().parent
LOCAL_CA_CERT = Path.cwd() / "certs" / "ca" / "ca.crt"

gitlab = requests.Session()
gitlab.headers.update({
    "PRIVATE-TOKEN": gitlab_token,
})
gitlab.verify = str(LOCAL_CA_CERT)

# Register and Connect GitLab Runner

In [ ]:
def create_instance_runner(description: str) -> dict:
    response = gitlab.post(
        f"{GITLAB_EXTERNAL_URL}/api/v4/user/runners",
        data={
            "description": description,
            "runner_type": "instance_type",
            "run_untagged": "true",
        },
        timeout=30,
    )

    response.raise_for_status()
    return response.json()


def register_runner(
        runner_token: str,
        container_name: str = "gitlab_runner"
) -> None:
    subprocess.run(
        [
            "docker",
            "exec",
            container_name,
            "gitlab-runner",
            "register",
            "--non-interactive",
            "--url", GITLAB_INTERNAL_URL,
            "--token", runner_token,
            "--executor", "docker",
            "--docker-image", "alpine:latest",
            "--clone-url", GITLAB_INTERNAL_URL,
            "--docker-network-mode", "cla321-network",
            "--docker-volumes", "/var/run/docker.sock:/var/run/docker.sock",
        ],
        check=True,
    )


instance_runner = create_instance_runner(
    description=f"test-stand-runner",
)

instance_runner_token: str = instance_runner["token"]

if not instance_runner_token.startswith("glrt-"):
    raise RuntimeError(f"Unexpected runner token: {instance_runner_token}")

register_runner(runner_token=instance_runner_token)

print(f"Runner registered: {instance_runner['id']}")

# Give access for Docker to the GitLab Container Registry

In [ ]:
%%bash

docker login registry.gitlab.localhost

# Connect GitLab Runner and K3d

In [ ]:
%%bash

docker network connect cla321-network k3d-dev-serverlb

# Create GitLab project

In [ ]:
def create_project(name: str, path: str) -> dict:
    response = gitlab.post(
        f"{GITLAB_EXTERNAL_URL}/api/v4/projects",
        json={
            "name": name,
            "path": path,
            "visibility": "private",
        },
        timeout=30,
    )

    response.raise_for_status()
    return response.json()


PROJECT_NAME = "demo"
PROJECT_PATH = "demo"

project = create_project(
    name=PROJECT_NAME,
    path=PROJECT_PATH,
)

print(f"project created: {project['web_url']}")

# Create GitLab project KUBECONFIG_CONTENT env var for GitLab Runner CI

In [ ]:
import yaml
from urllib.parse import quote

def set_project_variable(
    project_id: int,
    key: str,
    value: str,
) -> dict:
    data={
        "value": value,
        "variable_type": "env_var",
        "protected": "false",
        "masked": "false",
        "raw": "true",
    }


    response = gitlab.put(
        f"{GITLAB_EXTERNAL_URL}/api/v4/projects/{project_id}/variables/{key}",
        data=data,
        timeout=30,
    )

    if response.status_code == 404:
        response = gitlab.post(
            f"{GITLAB_EXTERNAL_URL}/api/v4/projects/{project_id}/variables",
            data={
                "key": key,
                **data,
            },
            timeout=30,
        )

    response.raise_for_status()
    return response.json()


def get_kubeconfig_content() -> str:
    result = subprocess.run(
        [
            "kubectl",
            "config",
            "view",
            "--raw",
            "--minify",
        ],
        check=True,
        capture_output=True,
        text=True,
    )

    config = yaml.safe_load(result.stdout)

    clusters = config.get("clusters")
    if not clusters:
        raise RuntimeError("Kubeconfig contains no clusters")

    if len(clusters) != 1:
        raise RuntimeError(f"Expected exactly one cluster, got {len(clusters)}")

    clusters[0]["cluster"]["server"] = (
        "https://k3d-dev-serverlb:6443"
    )

    return yaml.safe_dump(config, sort_keys=False)

def get_project(path: str) -> dict:
    project_path = quote(path, safe="")

    response = gitlab.get(
        f"{GITLAB_EXTERNAL_URL}/api/v4/projects/{project_path}",
        timeout=30,
    )

    response.raise_for_status()
    return response.json()

project = get_project(
    path="root/demo"
)

kubeconfig_content = get_kubeconfig_content()
set_project_variable(
    project_id=project["id"],
    key="KUBECONFIG_CONTENT",
    value=kubeconfig_content,
)

# Initialise GitLab project repository

In [ ]:
from pathlib import Path
import shutil
import subprocess
import tempfile
from urllib.parse import urlsplit, urlunsplit, quote

MONOREPO_PATH = Path("../monorepo")

repository_url = f"{GITLAB_EXTERNAL_URL}/root/demo"

parts = urlsplit(repository_url)

authenticated_repository_url = urlunsplit(
    (
        parts.scheme,
        f"oauth2:{gitlab_token}@{parts.netloc}",
        parts.path,
        parts.query,
        parts.fragment,
    )
)

with tempfile.TemporaryDirectory() as temp_dir:
    repo = Path(temp_dir) / "monorepo"

    shutil.copytree(
        MONOREPO_PATH,
        repo,
        ignore=shutil.ignore_patterns(
            ".git",
            ".idea",
            "target",
        ),
    )

    subprocess.run(
        ["git", "init", "-b", "main"],
        cwd=repo,
        check=True,
    )

    subprocess.run(
        ["git", "add", "--all"],
        cwd=repo,
        check=True,
    )

    subprocess.run(
        [
            "git",
            "-c",
            "user.name=Test Stand",
            "-c",
            "user.email=bootstrap@localhost",
            "commit",
            "-m",
            "Replace repository contents",
        ],
        cwd=repo,
        check=True,
    )

    subprocess.run(
        [
            "git",
            "remote",
            "add",
            "origin",
            authenticated_repository_url,
        ],
        cwd=repo,
        check=True,
    )

    subprocess.run(
        [
            "git",
            "push",
            "-u",
            "origin",
            "main",
        ],
        cwd=repo,
        check=True,
    )

# Give access to GitLab Registry for K3d

In [ ]:
import subprocess

result = subprocess.run(
    [
        "kubectl",
        "create",
        "secret",
        "docker-registry",
        "gitlab-registry",
        "--docker-server=registry.gitlab.localhost",
        "--docker-username=root",
        f"--docker-password={gitlab_token}",
        "--docker-email=unused@example.com",
        "--dry-run=client",
        "-o",
        "yaml"
    ],
    check=True,
    capture_output=True,
)

subprocess.run(
    [
        "kubectl",
        "apply",
        "-f",
        "-"
    ],
    input=result.stdout,
    check=True,
)

# Add SSH key to the root account

In [ ]:
import socket

public_key = Path.home().joinpath(".ssh", "id_ed25519.pub").read_text().strip()

response = gitlab.post(
    f"{GITLAB_EXTERNAL_URL}/api/v4/user/keys",
    data={
        "title": socket.gethostname(),
        "key": public_key
    },
    timeout=30,
)

response.raise_for_status()

print(response.json())

# Check that SSH works

In [ ]:
%%bash

ssh-keygen -R '[gitlab.localhost]:2222'
ssh -T -p 2222 git@gitlab.localhost